In [ ]:
# Student Name: Mohammad Abdalrouf Alnaghmoush
# Student ID: 2230004028


# Transformers


#### A Transformer is a deep learning architecture designed to process sequences of text using self-attention.

- Transformers are the foundation of many modern NLP models, including BERT, GPT, and T5.

- Before Transformers, we used models such as RNNs and LSTMs for NLP tasks.

<img src="https://drive.google.com/thumbnail?id=1unPqGaKftHxp37-F99Bg6hJzFbQdCGC7&sz=w1000" width="700">

In [ ]:
%pip -q install transformers torch matplotlib

## 1. Load a Transformer

We use **DistilBERT**, a  Transformer fine-tuned for positive/negative sentiment classification.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
torch.manual_seed(42)
torch.set_num_threads(2)
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, attn_implementation="eager")
model.eval()
print("Pretrained model:", model_name)
print("Transformer layers:", model.config.n_layers)
print("Attention heads per layer:", model.config.n_heads)
print("Hidden size:", model.config.dim)


<img src="https://drive.google.com/thumbnail?id=1v2dRK1KR_ydZOknjj8Xv8_wjOdXleDYs&sz=w1000" width="500">

## 2. Tokenizing the text

In [ ]:
sentence = "The movie was not good."

# Tokenization: converts text into token IDs
inputs = tokenizer(sentence, return_tensors="pt")

# [CLS] = start token, [SEP] = end token
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("Tokens:", tokens)

## 3. Embeddings and self-attention

Each token ID becomes a **vector (embedding)**. Inside each Transformer layer, **self-attention** lets a token use information from other tokens in the same sentence.

In [ ]:
import torch

# Enable attention weights
# Eager attention enabled in the model-loading cell

with torch.no_grad():
    embeddings = model.distilbert.embeddings(inputs["input_ids"])
    outputs = model(**inputs, output_attentions=True)

# Attention from the first layer and first head
attention = outputs.attentions[0][0, 0]

print("Embeddings shape:", embeddings.shape)
print("Attention shape:", attention.shape)
print("Tokens:", tokens)
print("Attention weights:\n", attention)

## 4. Classification: From Transformer to prediction

The Transformer produces contextual representations. The **classification head** turns them into two scores (*logits*). **Softmax** converts these into probabilities that sum to 1.

In [ ]:
# Get prediction scores from the Transformer
with torch.no_grad():
    logits = model(**inputs).logits

# Convert scores into probabilities
probs = torch.softmax(logits, dim=1)[0]

# Display the prediction
prediction = probs.argmax().item()

print("Prediction:", model.config.id2label[prediction])
print("Confidence:", round(probs[prediction].item(), 3))



### Task: Exploring Transformer Layers and Attention Heads

Load the pretrained **`bert-base-uncased`** model using Hugging Face Transformers.

1. Write code to print the number of Transformer layers and the number of attention heads per layer.
2. Calculate and print the total number of attention heads across all Transformer layers.
3. Using the sentence `"Transformers use self-attention to understand language."`, extract the attention weights from the **third Transformer layer and second attention head**.
4. Explain what the rows and columns of the attention matrix represent.

## Complete solution — Exploring Transformer Layers and Attention Heads
The following tasks use the actual **pretrained `bert-base-uncased` model** and the exact sentence specified in the question. The tokenizer and model weights are downloaded automatically on first execution; no external CSV dataset is required by this lab.


In [ ]:
from transformers import AutoModel, AutoTokenizer
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Tasks 1 and 2 — count layers and attention heads for pretrained BERT.
bert_name = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_name)
bert_model = AutoModel.from_pretrained(bert_name, attn_implementation="eager")
bert_model.eval()

layers = int(bert_model.config.num_hidden_layers)
heads_per_layer = int(bert_model.config.num_attention_heads)
total_heads = layers * heads_per_layer
print("Model:", bert_name)
print("Transformer layers:", layers)
print("Attention heads per layer:", heads_per_layer)
print("Total attention heads:", total_heads)
assert (layers, heads_per_layer, total_heads) == (12, 12, 144)


In [ ]:
# Task 3 — actual attention weights from the THIRD layer and SECOND head.
sentence_bert = "Transformers use self-attention to understand language."
tokenized_bert = bert_tokenizer(sentence_bert, return_tensors="pt")
tokens_bert = bert_tokenizer.convert_ids_to_tokens(tokenized_bert["input_ids"][0])

with torch.inference_mode():
    attention_outputs = bert_model(**tokenized_bert, output_attentions=True)

assert len(attention_outputs.attentions) == layers
# Zero-based indexes: [2] third layer; [0, 1] first item and second head.
head_attention = attention_outputs.attentions[2][0, 1].cpu().numpy()
assert head_attention.shape == (len(tokens_bert), len(tokens_bert))
assert np.allclose(head_attention.sum(axis=1), 1.0, atol=1e-5)

print("Sentence:", sentence_bert)
print("Tokens:", tokens_bert)
print("Layer 3 — Head 2 matrix dimensions:", head_attention.shape)
print("All row sums are approximately 1:", np.allclose(head_attention.sum(axis=1), 1.0, atol=1e-5))
matrix_table = pd.DataFrame(head_attention, index=tokens_bert, columns=tokens_bert)
matrix_table.index.name = "Query token (row)"
matrix_table.columns.name = "Key token (column)"
display(matrix_table.round(4))


In [ ]:
# Actual attention heatmap — saved as a visible image in the notebook outputs.
fig, ax = plt.subplots(figsize=(10, 8))
max_weight = float(head_attention.max())
heat = ax.imshow(head_attention, cmap="Blues", vmin=0, vmax=max_weight)
ax.set_xticks(range(len(tokens_bert)), labels=tokens_bert, rotation=55, ha="right")
ax.set_yticks(range(len(tokens_bert)), labels=tokens_bert)
ax.set_xlabel("Key tokens (columns)")
ax.set_ylabel("Query tokens (rows)")
ax.set_title("bert-base-uncased: Third Transformer Layer, Second Attention Head")
fig.colorbar(heat, ax=ax, label="Attention weight")
for row in range(len(tokens_bert)):
    for col in range(len(tokens_bert)):
        value = float(head_attention[row, col])
        ax.text(col, row, f"{value:.2f}", ha="center", va="center", fontsize=7,
                color="white" if value > max_weight * 0.55 else "black")
fig.tight_layout()
plt.show()


### Task 4 — What do the rows and columns of the attention matrix represent?

- **Rows = query tokens:** Each row represents a token whose attention distribution is being computed.
- **Columns = key tokens:** Each column represents a token that the query token can attend to.
- **Cell `[i,j]`:** The weight of the token in row `i` attending to the token in column `j`, in this particular head.
- **Normalization:** Each row's weights are nonnegative and sum to approximately 1 because a softmax normalizes across all key-token positions.
- **Special tokens:** BERT uses WordPiece tokenization; punctuation, hyphens, `[CLS]`, and `[SEP]` may appear as separate entries.

The matrix captures one head in the third layer, **not the whole model**. Attention weights should not be interpreted as a causal explanation for the classification result.


In [ ]:
# Task 4 — Inspect the relationship between query rows and key columns.
row_sums = head_attention.sum(axis=1)
print("Every row sums to 1:", bool(np.allclose(row_sums, 1, atol=1e-5)))
print("Top attended-to token (column) for each query token (row):")
for query, distribution in zip(tokens_bert, head_attention):
    best = int(distribution.argmax())
    print(f"{query:>15s} -> {tokens_bert[best]:<15s} | weight={distribution[best]:.4f}")
